In [1]:
!pip install ultralytics roboflow torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 38.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="iyLsqEim9nAoDa24N72g")
project = rf.workspace("industrial-engineer").project("cotton-disease-zrbov")
version = project.version(20)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Cotton-Disease-20 in yolov11:: 100%|██████████| 23420/23420 [00:03<00:00, 7377.55it/s]


In [3]:
import torch, torch.nn as nn, math
import ultralytics.nn.modules as m
import ultralytics.nn.tasks  as tasks
# from ultralytics.utils.torch_utils import make_divisible

# ----------------------------------------------------------
#  GhostConv  (cheap operation)
# ----------------------------------------------------------
class GhostConv(nn.Module):
    def __init__(self, c1, c2, k=1, s=1, g=1, act=True):
        super().__init__()
        c_ = c2 // 2
        # primary conv: c1 -> c_
        self.cv1 = nn.Conv2d(c1, c_, k, s, k//2, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_)
        # cheap operation: c_ -> c_  (depth-wise)
        self.cv2 = nn.Conv2d(c_, c_, 5, 1, 2, groups=c_, bias=False)
        self.bn2 = nn.BatchNorm2d(c_)
        self.act = nn.SiLU() if act else nn.Identity()

    def forward(self, x):
        y = self.act(self.bn1(self.cv1(x)))
        return torch.cat([y, self.act(self.bn2(self.cv2(y)))], 1)

# ----------------------------------------------------------
#  ECA (Efficient Channel Attention)
# ----------------------------------------------------------
class ECA(nn.Module):
    def __init__(self, c=None, gamma=2, b=1):
        super().__init__()
        self._c = c
        self._gamma = gamma
        self._b = b
        self._built = False

    def _build(self, c_in):
        t = int(abs((math.log(c_in, 2) + self._b) / self._gamma))
        k = t if t % 2 else t + 1
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)
        self._built = True

    def forward(self, x):
        if not self._built:
            self._build(x.size(1))
        y = self.pool(x)
        y = self.conv(y.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)
        y = torch.sigmoid(y)
        return x * y.expand_as(x)
# ----------------------------------------------------------
#  CBAM (Channel + Spatial)
# ----------------------------------------------------------
class CBAM(nn.Module):
    def __init__(self, c1=None, ratio=16, kernel_size=7):
        super().__init__()
        self._c1 = c1
        self._ratio = ratio
        self._ks = kernel_size
        self._built = False

    def _build(self, c_in):
        c_ = max(c_in // self._ratio, 16)
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c_in, c_, 1, bias=False), nn.SiLU(),
            nn.Conv2d(c_, c_in, 1, bias=False))
        self.sa = nn.Conv2d(2, 1, self._ks, padding=self._ks // 2, bias=False)
        self.sig = nn.Sigmoid()
        self._built = True

    def forward(self, x):
        if not self._built:
            self._build(x.size(1))
        ca = self.sig(self.ca(x))
        x = x * ca
        avg = torch.mean(x, dim=1, keepdim=True)
        maxv = torch.amax(x, dim=1, keepdim=True)
        sa = self.sig(self.sa(torch.cat([avg, maxv], 1)))
        return x * sa

# ----------------------------------------------------------
#  Bottleneck-CSP  (C3 in your drawing)
# ----------------------------------------------------------
class BottleneckCSP(nn.Module):
    def __init__(self, c1, c2, n=1, shortcut=True, g=1, e=0.5):
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = nn.Conv2d(c1, c_, 1, 1, bias=False)      # input = c1
        self.bn1 = nn.BatchNorm2d(c_)
        self.cv2 = nn.Conv2d(c1, c_, 1, 1, bias=False)      # input = c1
        self.bn2 = nn.BatchNorm2d(c_)
        self.cv3 = nn.Conv2d(c_, c2, 1, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()
        self.m = nn.Sequential(*(m.Bottleneck(c_, c_, shortcut, g, e=1.0) for _ in range(n)))

    def forward(self, x):
        return self.act(self.bn3(self.cv3(self.act(self.bn1(self.cv1(x))) + self.act(self.bn2(self.cv2(x))))))

# ----------------------------------------------------------
#  SPPF  (fast spatial pyramid)
# ----------------------------------------------------------
class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2
        self.cv1 = m.Conv(c1, c_, 1, 1)
        self.cv2 = m.Conv(c_ * 4, c2, 1, 1)
        self.m = nn.MaxPool2d(k, 1, k // 2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        return self.cv2(torch.cat([x, y1, y2, self.m(y2)], 1))

# ----------------------------------------------------------
#  register everything
# ----------------------------------------------------------
m.GhostConv = tasks.GhostConv = GhostConv
m.ECA = tasks.ECA = ECA
m.CBAM = tasks.CBAM = CBAM
m.BottleneckCSP = tasks.BottleneckCSP = BottleneckCSP
m.SPPF = tasks.SPPF = SPPF
print('✅ All custom modules registered')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ All custom modules registered


In [4]:
yaml_str = """
# path ------------------------------------------------------------------
train: /content/Cotton-Disease-20/train/images
val: /content/Cotton-Disease-20/valid/images
test: /content/Cotton-Disease-20/test/images

# model -----------------------------------------------------------------
nc: 1
names: ['Cotton Leaf Curl Virus']

depth_multiple: 0.33
width_multiple: 0.50

anchors:
  - [10,13, 16,30, 33,23]        # P3/8
  - [30,61, 62,45, 59,119]       # P4/16
  - [116,90, 156,198, 373,326]   # P5/32

# backbone (exact flow you drew) ----------------------------------------
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv,        [64, 6, 2, 2]]    #  0  P1/2
  - [-1, 1, GhostConv,  [128, 3, 2]]       #  1  P2/4
  - [-1, 2, BottleneckCSP,[128]]
  - [-1, 1, CBAM,        []]                #  3
  - [-1, 1, GhostConv,  [256, 3, 2]]       #  4  P3/8
  - [-1, 4, BottleneckCSP,[256]]
  - [-1, 1, CBAM,        []]                #  6
  - [-1, 1, ECA,         []]                #  7
  - [-1, 1, GhostConv,  [512, 3, 2]]       #  8  P4/16
  - [-1, 4, BottleneckCSP,[512]]
  - [-1, 1, CBAM,        []]                # 10
  - [-1, 1, ECA,         []]                # 11
  - [-1, 1, GhostConv,  [512, 3, 2]]       # 12 P5/32
  - [-1, 2, BottleneckCSP,[512]]
  - [-1, 1, CBAM,        []]                # 14
  - [-1, 1, SPPF,        [512, 5]]          # 15

# head (PANet) ----------------------------------------------------------
head:
  - [-1, 1, GhostConv,  [256, 1, 1]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 9], 1, Concat, [1]]              # 18
  - [-1, 2, BottleneckCSP,[256]]
  - [-1, 1, CBAM,        []]                # 20
  - [-1, 1, GhostConv,  [128, 1, 1]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 6], 1, Concat, [1]]              # 23
  - [-1, 2, BottleneckCSP,[128]]
  - [-1, 1, CBAM,        []]                # 25
  # down path
  - [-1, 1, GhostConv,  [128, 3, 2]]
  - [[-1, 20], 1, Concat, [1]]             # 27
  - [-1, 2, BottleneckCSP,[256]]
  - [-1, 1, CBAM,        []]                # 29
  - [-1, 1, GhostConv,  [256, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]             # 31
  - [-1, 2, BottleneckCSP,[512]]
  - [-1, 1, CBAM,        []]                # 33
  # detect
  - [[25, 29, 33], 1, Detect, [1]]         # 34  P3/8, P4/16, P5/32
"""
with open('yolo11_CBAM.yaml', 'w') as f:
    f.write(yaml_str.strip())
print('✅ CBAM yaml saved')

✅ CBAM yaml saved


In [6]:
from ultralytics import YOLO
model = YOLO('/content/yolo11_CBAM.yaml')        # builds OK
model.info()

YOLO11_CBAM summary: 225 layers, 3,191,565 parameters, 3,191,549 gradients, 7.5 GFLOPs


(225, 3191565, 3191549, 7.4654944)

In [7]:
                                       # sanity check

model.train(data='/content/yolo11_CBAM.yaml',
            epochs=50,
            imgsz=640,
            batch=64,
            optimizer='AdamW',
            amp=True,
            project='yolo11-custom',
            name='exp')

Ultralytics 8.3.218 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo11_CBAM.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolo11_CBAM.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=exp, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, perspective=0.0, plots=T

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b69699e2150>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 